## Imports

In [123]:
from sklearn.base import BaseEstimator, ClassifierMixin
from itertools import product
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from category_encoders import CountEncoder
from sklearn.model_selection import GridSearchCV
import scipy
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.pipeline import Pipeline
import shap
import optuna

### 1. Download data from Don’tGetKicked competition.


In [124]:
df = pd.read_csv(filepath_or_buffer='data/training.csv', index_col=0, parse_dates=['PurchDate'])
df['PRIMEUNIT'] = df['PRIMEUNIT'].fillna('MISSING')
df['AUCGUART'] = df['AUCGUART'].fillna('MISSING')
df.isna().sum()

IsBadBuy                                0
PurchDate                               0
Auction                                 0
VehYear                                 0
VehicleAge                              0
Make                                    0
Model                                   0
Trim                                 2360
SubModel                                8
Color                                   8
Transmission                            9
WheelTypeID                          3169
WheelType                            3174
VehOdo                                  0
Nationality                             5
Size                                    5
TopThreeAmericanName                    5
MMRAcquisitionAuctionAveragePrice      18
MMRAcquisitionAuctionCleanPrice        18
MMRAcquisitionRetailAveragePrice       18
MMRAcquisitonRetailCleanPrice          18
MMRCurrentAuctionAveragePrice         315
MMRCurrentAuctionCleanPrice           315
MMRCurrentRetailAveragePrice      

In [125]:
y = df['IsBadBuy']
X = df[df.columns[1:]]
X

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
1,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,...,11597.0,12409.0,MISSING,MISSING,21973,33619,FL,7100.0,0,1113
2,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,...,11374.0,12791.0,MISSING,MISSING,19638,33619,FL,7600.0,0,1053
3,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,...,7146.0,8702.0,MISSING,MISSING,19638,33619,FL,4900.0,0,1389
4,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,...,4375.0,5518.0,MISSING,MISSING,19638,33619,FL,4100.0,0,630
5,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,...,6739.0,7911.0,MISSING,MISSING,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73010,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,...,4836.0,5937.0,MISSING,MISSING,18111,30212,GA,4200.0,0,993
73011,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,...,10151.0,11652.0,MISSING,MISSING,18881,30212,GA,6200.0,0,1038
73012,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,...,11831.0,14402.0,MISSING,MISSING,18111,30212,GA,8200.0,0,1893


### 2. Design the train/validation/test split. Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set. Don’t use the test dataset until the end!

In [126]:
def my_train_valid_test_date_split(X, y, validation_date, test_date):
    train_mask = X['PurchDate'] < validation_date
    valid_mask = (X['PurchDate'] >= validation_date) & (X['PurchDate'] <= test_date)
    test_mask = X['PurchDate'] > test_date

    X_train = X[train_mask]
    X_valid = X[valid_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_valid = y[valid_mask]
    y_test = y[test_mask]

    X_train = X_train.drop(labels=['PurchDate'], axis=1)
    X_valid = X_valid.drop(labels=['PurchDate'], axis=1)
    X_test = X_test.drop(labels=['PurchDate'], axis=1)

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [127]:
first_date = df['PurchDate'].quantile(0.34)
second_date = df['PurchDate'].quantile(0.67)
print(first_date)
print(second_date)

2009-09-17 00:00:00
2010-05-18 00:00:00


In [128]:
X_train, X_valid, X_test, y_train, y_valid, y_test = my_train_valid_test_date_split(X, y, first_date, second_date)
print(len(X_train), len(X_valid), len(X_test))

24695 24372 23916


In [129]:
fill_values = {
    'WheelType':    X_train['WheelType'].mode()[0],
    'WheelTypeID':  X_train['WheelTypeID'].mode()[0],
    'Trim':         X_train['Trim'].mode()[0],
    'SubModel':     X_train['SubModel'].mode()[0],
    'Color':        X_train['Color'].mode()[0],
    'Transmission': X_train['Transmission'].mode()[0],
    'Nationality':  X_train['Nationality'].mode()[0],
    'Size':         X_train['Size'].mode()[0],
    'TopThreeAmericanName': X_train['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train['MMRCurrentRetailCleanPrice'].median(),
}

X_train.fillna(fill_values, inplace=True)
X_valid.fillna(fill_values, inplace=True)
X_test.fillna(fill_values, inplace=True)

### 1. Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html


In [130]:
categorical_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
    'Transmission', 'WheelType', 'Nationality', 'Size',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST',
    'WheelTypeID', 'BYRNO', 'VNZIP1'
]

In [131]:
encoder = CountEncoder(cols=categorical_cols)
X_train = encoder.fit_transform(X_train)
X_valid = encoder.transform(X_valid)
X_test = encoder.transform(X_test)
X_test

,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
266,5076,2006,4,5743.0,0.0,3039.0,142.0,109,24008,10788.0,...,12427.0,13850.0,24693.0,24693.0,545.0,458.0,3452.0,6800.0,0,1630
267,5076,2004,6,5743.0,0.0,183.0,493.0,1921,24008,13666.0,...,9497.0,11161.0,24693.0,24693.0,913.0,458.0,3452.0,8600.0,0,1020
268,5076,2006,4,2832.0,3.0,5359.0,4991.0,2048,24008,10788.0,...,8667.0,9390.0,24693.0,24693.0,913.0,458.0,3452.0,6150.0,0,1215
269,5076,2007,3,656.0,9.0,210.0,179.0,729,24008,10788.0,...,9792.0,10587.0,24693.0,24693.0,913.0,458.0,3452.0,5800.0,0,728
270,5076,2002,8,287.0,1.0,233.0,106.0,2048,24008,13666.0,...,8950.0,10403.0,24693.0,24693.0,545.0,458.0,3452.0,7600.0,0,1543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72415,5076,2004,6,656.0,0.0,47.0,47.0,2504,24008,241.0,...,6364.0,7176.0,24693.0,24693.0,0.0,409.0,4615.0,4105.0,0,2891
72416,5076,2003,7,4852.0,2.0,434.0,12.0,2048,24008,13666.0,...,7853.0,8917.0,0.0,0.0,0.0,409.0,4615.0,4490.0,0,1930
72417,5076,2002,8,4323.0,2.0,3484.0,1448.0,3486,24008,13666.0,...,5411.0,6523.0,0.0,0.0,0.0,409.0,4615.0,3690.0,0,1455


In [132]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

### Train LogisticRegression, GaussianNB, KNN from sklearn on the training dataset and check the quality of your algorithms on the validation dataset. The dependent variable (IsBadBuy) is binary. Don't forget to normalize your datasets before training your models.

You must get at least 0.15 Gini score (the best of all three). Which algorithm performs better? And why?

In [133]:
def my_roc_auc(y_class, y_pred):
    positive = (y_class == 1).sum()
    negative = (y_class == 0).sum()

    sorted_idx = np.argsort(y_pred)[::-1]
    y_class_sorted = y_class.iloc[sorted_idx]
    TP = 0
    FP = 0
    points = [(0, 0)]

    for klass in y_class_sorted:
        if klass == 1:
            TP += 1
        else:
            FP += 1
        TPR = TP / positive
        FPR = FP / negative
        points.append((FPR, TPR))

    auc = 0
    for i in range(len(points) - 1):
        x1, y1 = points[i]
        x2, y2 = points[i + 1]
        width = x2 - x1
        height = (y2 + y1) / 2
        auc += width * height

    return auc

In [134]:
def gini(auc_score):
    gini = abs(2 * auc_score - 1)
    return gini

In [135]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)
X_test

array([[0.        , 0.71428571, 0.42857143, ..., 0.14953374, 0.        ,
        0.16600341],
       [0.        , 0.42857143, 0.71428571, ..., 0.18912202, 0.        ,
        0.07930642],
       [0.        , 0.71428571, 0.42857143, ..., 0.13523797, 0.        ,
        0.10702103],
       ...,
       [0.        , 0.14285714, 1.        , ..., 0.08113398, 0.        ,
        0.14113132],
       [0.        , 0.42857143, 0.71428571, ..., 0.10092813, 0.        ,
        0.14539511],
       [0.        , 0.57142857, 0.57142857, ..., 0.09213073, 0.        ,
        0.05557135]], shape=(23916, 31))

In [136]:
logreg = LogisticRegression(random_state=21)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_valid)[:, 1]
sklearn_auc = roc_auc_score(y_valid, logreg_proba)
gini(sklearn_auc)

0.3235988028273682

In [137]:
Gauss = GaussianNB()
Gauss.fit(X_train, y_train)
Gauss_predict = Gauss.predict(X_valid)
sklearn_auc = roc_auc_score(Gauss_predict, y_valid)
gini(sklearn_auc)

0.4035965020322696

In [138]:
KNN = KNeighborsClassifier()
KNN.fit(X_train, y_train)
KNN_predict = KNN.predict(X_valid)
sklearn_auc = roc_auc_score(KNN_predict, y_valid)
gini(sklearn_auc)

0.10847745940524267

### Implement Gini score calculation. You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AUC calculation. Check if your metric is approximately equal to abs(2*sklearn.metrics.roc_auc_score - 1).

In [139]:
logreg = LogisticRegression(random_state=21)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_valid)[:, 1]
sklearn_auc = my_roc_auc(y_valid, logreg_proba)
gini(sklearn_auc)

np.float64(0.3235988028273469)

In [140]:
Gauss = GaussianNB()
Gauss.fit(X_train, y_train)
Gauss_proba = Gauss.predict_proba(X_valid)
sklearn_auc = roc_auc_score(y_valid, Gauss_proba[:, 1])
gini(sklearn_auc)

0.3021863043041242

In [141]:
KNN = KNeighborsClassifier()
KNN.fit(X_train, y_train)
KNN_predict_proba = KNN.predict_proba(X_valid)
sklearn_auc = roc_auc_score(y_valid, KNN_predict_proba[:, 1])
gini(sklearn_auc)

0.13737394876419362

### Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers. For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?

In [142]:
def sigmoid(x):
    return np.where(x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

In [143]:
class SGDLogisticRegression(ClassifierMixin, BaseEstimator):
    def __init__(self, epoch, learning_rate=0.01, l1=0, l2=0):
        self.bias = 0
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.l1 = l1
        self.l2 = l2
        self.batch_size = 64

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.classes_ = np.unique(y)
        self.weights = np.zeros(X.shape[1])
        for epoc in range(self.epoch):
            indexes = np.random.permutation(len(X))
            for i in range(0, len(indexes), self.batch_size):
                idx = indexes[i : i + self.batch_size]
                xi = X[idx]
                yi = y[idx]
                z = self.bias + (xi @ self.weights)
                prediction = sigmoid(z)
                errors = yi - prediction
                grad = (xi.T @ (- errors) / len(xi) + (2 * self.l2 * self.weights) / len(xi)) + (self.l1 * np.sign(self.weights) / len(xi))
                self.weights = self.weights - self.learning_rate * grad
                self.bias = self.bias - self.learning_rate * np.mean(-errors)
        return self

    def predict(self, X):
        X = np.array(X)
        proba = sigmoid(self.bias + (X @ self.weights))
        return (proba >= 0.5).astype(int)

    def predict_proba(self, X):
        X = np.array(X)
        proba_positive = sigmoid(self.bias + (X @ self.weights))
        return np.column_stack([1 - proba_positive, proba_positive])

In [144]:
class MyGaussianNB:
    def __init__(self):
        self.priors = {}
        self.means = {}
        self.vars = {}

    def fit(self, X, y):
        idx0 = (y==0)
        idx1 = (y==1)
        X_0 = X[idx0]
        X_1 = X[idx1]

        self.priors[0] = len(X_0) / len(X)
        self.priors[1] = len(X_1) / len(X)
        self.means[0] = np.mean(X_0, axis=0)
        self.means[1] = np.mean(X_1, axis=0)
        self.vars[0] = np.var(X_0, axis=0) + 1e-9
        self.vars[1] = np.var(X_1, axis=0) + 1e-9

    def predict(self, X):
        predicts = []
        for x in X:
            score0 = np.log(self.priors[0]) + np.sum(-0.5 * np.log(2 * np.pi * self.vars[0]) - (x - self.means[0]) ** 2 / (2 * self.vars[0]))
            score1 = np.log(self.priors[1]) + np.sum(-0.5 * np.log(2 * np.pi * self.vars[1]) - (x - self.means[1]) ** 2 / (2 * self.vars[1]))
            predicts.append(0 if score0 > score1 else 1)
        return np.array(predicts)
    def predict_proba(self, X):
        predicts = []
        for x in X:
            score0 = np.log(self.priors[0]) + np.sum(-0.5 * np.log(2 * np.pi * self.vars[0]) - (x - self.means[0]) ** 2 / (2 * self.vars[0]))
            score1 = np.log(self.priors[1]) + np.sum(-0.5 * np.log(2 * np.pi * self.vars[1]) - (x - self.means[1]) ** 2 / (2 * self.vars[1]))
            max_score = max(score0, score1)
            exp0 = np.exp(score0 - max_score)
            exp1 = np.exp(score1 - max_score)
            total = exp0 + exp1
            prob0 = exp0 / total
            prob1 = exp1 / total

            predicts.append((prob0, prob1))
        return np.array(predicts)

In [145]:
class MyKNN:
    def __init__(self, K):
        self.KNeighbors = K

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((x - self.X_train) ** 2, axis=1))
            sorted_idxs = np.argsort(distances)
            neighbors = self.y_train[sorted_idxs[:self.KNeighbors]]
            classes, counts = np.unique(neighbors, return_counts=True)
            predictions.append(classes[np.argmax(counts)])
        return np.array(predictions)

    def predict_proba(self, X):
        probas = []
        for x in X:
            distances = np.sqrt(np.sum((x - self.X_train) ** 2, axis=1))
            sorted_idxs = np.argsort(distances)
            neighbors = self.y_train[sorted_idxs[:self.KNeighbors]]
            prob1 = np.sum(neighbors == 1) / self.KNeighbors
            prob0 = 1 - prob1
            probas.append([prob0, prob1])
        return np.array(probas)


In [146]:
mylogreg = SGDLogisticRegression(epoch=500)
mylogreg.fit(X_train, y_train)
mylogreg_proba = mylogreg.predict_proba(X_valid)[:, 1]
my_auc = my_roc_auc(y_valid, mylogreg_proba)
gini(my_auc)

np.float64(0.30501618238603156)

In [147]:
myGauss = MyGaussianNB()
myGauss.fit(X_train, y_train)
myGauss_proba = myGauss.predict_proba(X_valid)
my_auc = roc_auc_score(y_valid, myGauss_proba[:, 1])
gini(my_auc)

0.30218621498421294

In [148]:
myKNN = MyKNN(11)
myKNN.fit(X_train, y_train)
myKNN_predict_proba = myKNN.predict_proba(X_valid)
my_auc = roc_auc_score(y_valid, myKNN_predict_proba[:, 1])
gini(my_auc)

0.1910384827095708

### 7. Try to create non-linear features, for example:

fractions: feature1/feature2
groupby features: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())

Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?

In [149]:
df['price_ratio'] = df['VehBCost'] / df['MMRCurrentRetailAveragePrice']
df['depreciation'] = df['MMRAcquisitionRetailAveragePrice'] / df['MMRCurrentRetailAveragePrice']
df['odo_per_year'] = df['VehOdo'] / (df['VehicleAge'] + 1)
df['warranty_ratio'] = df['WarrantyCost'] / df['VehBCost']

In [150]:
df['make_mean_price'] = df['Make'].map(df.groupby('Make')['VehBCost'].mean())
df['model_mean_odo'] = df['Model'].map(df.groupby('Model')['VehOdo'].mean())

In [151]:
y = df['IsBadBuy']
X = df[df.columns[1:]]
X

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,VNST,VehBCost,IsOnlineSale,WarrantyCost,price_ratio,depreciation,odo_per_year,warranty_ratio,make_mean_price,model_mean_odo
RefId,,,,,,,,,,,,,,,,,,,,,
1,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,...,FL,7100.0,0,1113,0.612227,1.003363,22261.500000,0.156761,7049.540603,71126.792079
2,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,...,FL,7600.0,0,1053,0.668191,0.958062,15598.833333,0.138553,7046.992268,76037.878292
3,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,...,FL,4900.0,0,1389,0.685698,0.971592,14761.400000,0.283469,7046.992268,66739.055249
4,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,...,FL,4100.0,0,630,0.937143,1.064686,10936.166667,0.153659,7046.992268,73164.542553
5,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,...,FL,4000.0,0,1020,0.593560,1.146016,13873.400000,0.255000,6403.081203,75596.863586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73010,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,...,GA,4200.0,0,993,0.868486,0.549214,5026.000000,0.236429,6231.858160,76566.710383
73011,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,...,GA,6200.0,0,1038,0.610777,0.732046,23919.666667,0.167419,6834.965065,70811.884898
73012,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,...,GA,8200.0,0,1893,0.693094,0.822331,17700.000000,0.230854,7915.838662,75251.996350


In [152]:
X_train, X_valid, X_test, y_train, y_valid, y_test = my_train_valid_test_date_split(X, y, first_date, second_date)
print(len(X_train), len(X_valid), len(X_test))

24695 24372 23916


In [153]:
feature_names = X_train.columns.tolist()

In [154]:
coeffs_args_by_usefullness = np.argsort(abs(logreg.coef_)[0])
feature_names_sorted = np.array(feature_names)[coeffs_args_by_usefullness]
print(feature_names_sorted)
sorted(abs(logreg.coef_)[0])

['IsOnlineSale' 'Color' 'MMRAcquisitionAuctionAveragePrice' 'VNZIP1'
 'MMRAcquisitionRetailAveragePrice' 'Make' 'Size'
 'MMRCurrentAuctionAveragePrice' 'Trim' 'VNST' 'BYRNO' 'SubModel'
 'WarrantyCost' 'MMRCurrentAuctionCleanPrice'
 'MMRAcquisitionAuctionCleanPrice' 'Nationality' 'Auction'
 'MMRAcquisitonRetailCleanPrice' 'VehicleAge' 'TopThreeAmericanName'
 'Transmission' 'Model' 'WheelTypeID' 'WheelType' 'VehYear'
 'MMRCurrentRetailCleanPrice' 'AUCGUART' 'PRIMEUNIT' 'VehOdo'
 'MMRCurrentRetailAveragePrice' 'VehBCost']


[np.float64(0.0),
 np.float64(0.008670066263383649),
 np.float64(0.024415031818453822),
 np.float64(0.03999440002754176),
 np.float64(0.05956292711663033),
 np.float64(0.06304027554238906),
 np.float64(0.06438362485955999),
 np.float64(0.1373263299764807),
 np.float64(0.15078913386821657),
 np.float64(0.19616922678715995),
 np.float64(0.19772258895714787),
 np.float64(0.24457421532185855),
 np.float64(0.3105336329166909),
 np.float64(0.3126181691434077),
 np.float64(0.32560251986163),
 np.float64(0.33222807767620577),
 np.float64(0.34457185386618033),
 np.float64(0.35059613055321787),
 np.float64(0.4420137364573962),
 np.float64(0.4436589344088505),
 np.float64(0.44699354859847296),
 np.float64(0.6215053850755542),
 np.float64(0.7539075331127236),
 np.float64(0.7539075331127236),
 np.float64(0.8467355213356474),
 np.float64(1.074700996556587),
 np.float64(1.2454085933433323),
 np.float64(1.2454085933433323),
 np.float64(1.2716600907094808),
 np.float64(1.3452121001665578),
 np.float64(

In [155]:
best_features = feature_names_sorted[15:]
best_features

array(['Nationality', 'Auction', 'MMRAcquisitonRetailCleanPrice',
       'VehicleAge', 'TopThreeAmericanName', 'Transmission', 'Model',
       'WheelTypeID', 'WheelType', 'VehYear',
       'MMRCurrentRetailCleanPrice', 'AUCGUART', 'PRIMEUNIT', 'VehOdo',
       'MMRCurrentRetailAveragePrice', 'VehBCost'], dtype='<U33')

In [156]:
X_train_new = X_train[best_features]
X_valid_new = X_valid[best_features]
X_test_new = X_test[best_features]

In [157]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

In [158]:
fill_values = {
    'price_ratio':   X_train['price_ratio'].median(),
    'depreciation':  X_train['depreciation'].median(),
    'WheelType':    X_train['WheelType'].mode()[0],
    'WheelTypeID':  X_train['WheelTypeID'].mode()[0],
    'Trim':         X_train['Trim'].mode()[0],
    'SubModel':     X_train['SubModel'].mode()[0],
    'Color':        X_train['Color'].mode()[0],
    'Transmission': X_train['Transmission'].mode()[0],
    'Nationality':  X_train['Nationality'].mode()[0],
    'Size':         X_train['Size'].mode()[0],
    'TopThreeAmericanName': X_train['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train['MMRCurrentRetailCleanPrice'].median(),
}

X_train.fillna(fill_values, inplace=True)
X_valid.fillna(fill_values, inplace=True)
X_test.fillna(fill_values, inplace=True)

In [159]:
categorical_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
    'Transmission', 'WheelType', 'Nationality', 'Size',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST',
    'WheelTypeID', 'BYRNO', 'VNZIP1'
]

In [160]:
encoder = CountEncoder(cols=categorical_cols)
X_train = encoder.fit_transform(X_train)
X_valid = encoder.transform(X_valid)
X_test = encoder.transform(X_test)
X_test

,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,VNST,VehBCost,IsOnlineSale,WarrantyCost,price_ratio,depreciation,odo_per_year,warranty_ratio,make_mean_price,model_mean_odo
RefId,,,,,,,,,,,,,,,,,,,,,
266,5076,2006,4,5743.0,0.0,3039.0,142.0,109,24008,10788.0,...,3452.0,6800.0,0,1630,0.547196,0.932325,18354.600000,0.239706,6834.965065,77773.741742
267,5076,2004,6,5743.0,0.0,183.0,493.0,1921,24008,13666.0,...,3452.0,8600.0,0,1020,0.905549,1.026429,8895.857143,0.118605,6834.965065,79303.010060
268,5076,2006,4,2832.0,3.0,5359.0,4991.0,2048,24008,10788.0,...,3452.0,6150.0,0,1215,0.709588,0.978770,13313.600000,0.197561,6507.348541,64277.384285
269,5076,2007,3,656.0,9.0,210.0,179.0,729,24008,10788.0,...,3452.0,5800.0,0,728,0.592320,1.011949,15769.750000,0.125517,5606.276237,70762.640278
270,5076,2002,8,287.0,1.0,233.0,106.0,2048,24008,13666.0,...,3452.0,7600.0,0,1543,0.849162,0.918212,9128.666667,0.203026,8342.241140,78360.379310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72415,5076,2004,6,656.0,0.0,47.0,47.0,2504,24008,241.0,...,4615.0,4105.0,0,2891,0.645035,1.032684,14833.428571,0.704263,5606.276237,73753.793296
72416,5076,2003,7,4852.0,2.0,434.0,12.0,2048,24008,13666.0,...,4615.0,4490.0,0,1930,0.571756,0.829110,11553.375000,0.429844,7046.992268,77455.570248
72417,5076,2002,8,4323.0,2.0,3484.0,1448.0,3486,24008,13666.0,...,4615.0,3690.0,0,1455,0.681944,0.881168,8730.777778,0.394309,6403.081203,75596.863586


In [161]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

In [162]:
mylogreg = SGDLogisticRegression(epoch=500)
mylogreg.fit(X_train, y_train)
mylogreg_proba = mylogreg.predict_proba(X_valid)[:, 1]
my_auc = my_roc_auc(y_valid, mylogreg_proba)
gini(my_auc)

np.float64(0.34398559615205615)

In [163]:
myGauss = MyGaussianNB()
myGauss.fit(X_train, y_train)
myGauss_proba = myGauss.predict_proba(X_valid)
my_auc = roc_auc_score(y_valid, myGauss_proba[:, 1])
gini(my_auc)

0.32791859657221156

In [164]:
KNN = KNeighborsClassifier()
KNN.fit(X_train, y_train)
KNN_predict_proba = KNN.predict_proba(X_valid)
sklearn_auc = roc_auc_score(y_valid, KNN_predict_proba[:, 1])
gini(sklearn_auc)

0.1646541402041546

### 8. Determine the best features for the problem using the coefficients of the logistic model. Try to eliminate useless features by hand and by L1 regularization. Which approach is better in terms of Gini score?

In [165]:
best_features

array(['Nationality', 'Auction', 'MMRAcquisitonRetailCleanPrice',
       'VehicleAge', 'TopThreeAmericanName', 'Transmission', 'Model',
       'WheelTypeID', 'WheelType', 'VehYear',
       'MMRCurrentRetailCleanPrice', 'AUCGUART', 'PRIMEUNIT', 'VehOdo',
       'MMRCurrentRetailAveragePrice', 'VehBCost'], dtype='<U33')

In [166]:
fill_values = {
    'WheelType':    X_train_new['WheelType'].mode()[0],
    'WheelTypeID':  X_train_new['WheelTypeID'].mode()[0],
    'Transmission': X_train_new['Transmission'].mode()[0],
    'Nationality':  X_train_new['Nationality'].mode()[0],
    'TopThreeAmericanName': X_train_new['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitonRetailCleanPrice':     X_train_new['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train_new['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train_new['MMRCurrentRetailCleanPrice'].median(),
}

X_train_new.fillna(fill_values, inplace=True)
X_valid_new.fillna(fill_values, inplace=True)
X_test_new.fillna(fill_values, inplace=True)

In [167]:
categorical_cols = [
    'Auction', 'Model',
    'Transmission', 'WheelType', 'Nationality',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART',
    'WheelTypeID'
]

In [168]:
X_train_new.columns

Index(['Nationality', 'Auction', 'MMRAcquisitonRetailCleanPrice', 'VehicleAge',
       'TopThreeAmericanName', 'Transmission', 'Model', 'WheelTypeID',
       'WheelType', 'VehYear', 'MMRCurrentRetailCleanPrice', 'AUCGUART',
       'PRIMEUNIT', 'VehOdo', 'MMRCurrentRetailAveragePrice', 'VehBCost'],
      dtype='object')

In [169]:
encoder = CountEncoder(cols=categorical_cols)
X_train_new = encoder.fit_transform(X_train_new)
X_valid_new = encoder.transform(X_valid_new)
X_test_new = encoder.transform(X_test_new)

In [170]:
scaler = StandardScaler()
X_train_new = scaler.fit_transform(X_train_new)
X_valid_new = scaler.transform(X_valid_new)
X_test_new = scaler.transform(X_test_new)

In [171]:
mylogreg = SGDLogisticRegression(epoch=500, l1=1)
mylogreg.fit(X_train_new, y_train)
mylogreg_proba = mylogreg.predict_proba(X_valid_new)[:, 1]
my_auc = my_roc_auc(y_valid, mylogreg_proba)
gini(my_auc)

np.float64(0.2974711806329571)

In [172]:
myGauss = MyGaussianNB()
myGauss.fit(X_train_new, y_train)
myGauss_proba = myGauss.predict_proba(X_valid_new)
my_auc = roc_auc_score(y_valid, myGauss_proba[:, 1])
gini(my_auc)

0.32682002121288334

In [173]:
KNN = KNeighborsClassifier()
KNN.fit(X_train_new, y_train)
KNN_predict_proba = KNN.predict_proba(X_valid_new)
sklearn_auc = roc_auc_score(y_valid, KNN_predict_proba[:, 1])
gini(sklearn_auc)

0.1470349689833632

### 9. Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. Which hyperparameters have the most impact?

##### best model - linreg with polynomial features.

In [174]:
base_columns = [c for c in df.columns if c not in
                ['price_ratio', 'depreciation', 'odo_per_year',
                 'warranty_ratio', 'make_mean_price', 'model_mean_odo']]

In [175]:
y = df['IsBadBuy']
X = df[base_columns[1:]]

X_train_raw, X_valid_raw, X_test_raw, y_train, y_valid, y_test = my_train_valid_test_date_split(
    X, y, first_date, second_date
)

fill_values_std = {
    'WheelType':    X_train_raw['WheelType'].mode()[0],
    'WheelTypeID':  X_train_raw['WheelTypeID'].mode()[0],
    'Trim':         X_train_raw['Trim'].mode()[0],
    'SubModel':     X_train_raw['SubModel'].mode()[0],
    'Color':        X_train_raw['Color'].mode()[0],
    'Transmission': X_train_raw['Transmission'].mode()[0],
    'Nationality':  X_train_raw['Nationality'].mode()[0],
    'Size':         X_train_raw['Size'].mode()[0],
    'TopThreeAmericanName': X_train_raw['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train_raw['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train_raw['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train_raw['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train_raw['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train_raw['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train_raw['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train_raw['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train_raw['MMRCurrentRetailCleanPrice'].median(),
}

X_train_raw.fillna(fill_values_std, inplace=True)
X_valid_raw.fillna(fill_values_std, inplace=True)
X_test_raw.fillna(fill_values_std, inplace=True)

categorical_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
    'Transmission', 'WheelType', 'Nationality', 'Size',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST',
    'WheelTypeID', 'BYRNO', 'VNZIP1'
]

encoder_std = CountEncoder(cols=categorical_cols)
X_train_enc_std = encoder_std.fit_transform(X_train_raw)
X_valid_enc_std = encoder_std.transform(X_valid_raw)
X_test_enc_std  = encoder_std.transform(X_test_raw)

scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train_enc_std)
X_valid_std = scaler_std.transform(X_valid_enc_std)
X_test_std  = scaler_std.transform(X_test_enc_std)

In [176]:
df['price_ratio'] = df['VehBCost'] / df['MMRCurrentRetailAveragePrice']
df['depreciation'] = df['MMRAcquisitionRetailAveragePrice'] / df['MMRCurrentRetailAveragePrice']
df['odo_per_year'] = df['VehOdo'] / (df['VehicleAge'] + 1)
df['warranty_ratio'] = df['WarrantyCost'] / df['VehBCost']
df['make_mean_price'] = df['Make'].map(df.groupby('Make')['VehBCost'].mean())
df['model_mean_odo'] = df['Model'].map(df.groupby('Model')['VehOdo'].mean())

y = df['IsBadBuy']
X = df[df.columns[1:]]

X_train_poly_raw, X_valid_poly_raw, X_test_poly_raw, y_train, y_valid, y_test = my_train_valid_test_date_split(
    X, y, first_date, second_date
)

X_train_poly_raw = X_train_poly_raw.replace([np.inf, -np.inf], np.nan)
X_valid_poly_raw = X_valid_poly_raw.replace([np.inf, -np.inf], np.nan)
X_test_poly_raw  = X_test_poly_raw.replace([np.inf, -np.inf], np.nan)

fill_values_poly = {
    'price_ratio':   X_train_poly_raw['price_ratio'].median(),
    'depreciation':  X_train_poly_raw['depreciation'].median(),
    'WheelType':    X_train_poly_raw['WheelType'].mode()[0],
    'WheelTypeID':  X_train_poly_raw['WheelTypeID'].mode()[0],
    'Trim':         X_train_poly_raw['Trim'].mode()[0],
    'SubModel':     X_train_poly_raw['SubModel'].mode()[0],
    'Color':        X_train_poly_raw['Color'].mode()[0],
    'Transmission': X_train_poly_raw['Transmission'].mode()[0],
    'Nationality':  X_train_poly_raw['Nationality'].mode()[0],
    'Size':         X_train_poly_raw['Size'].mode()[0],
    'TopThreeAmericanName': X_train_poly_raw['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train_poly_raw['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train_poly_raw['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train_poly_raw['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train_poly_raw['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train_poly_raw['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train_poly_raw['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train_poly_raw['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train_poly_raw['MMRCurrentRetailCleanPrice'].median(),
}

X_train_poly_raw.fillna(fill_values_poly, inplace=True)
X_valid_poly_raw.fillna(fill_values_poly, inplace=True)
X_test_poly_raw.fillna(fill_values_poly, inplace=True)

encoder_poly = CountEncoder(cols=categorical_cols)
X_train_enc_poly = encoder_poly.fit_transform(X_train_poly_raw)
X_valid_enc_poly = encoder_poly.transform(X_valid_poly_raw)
X_test_enc_poly  = encoder_poly.transform(X_test_poly_raw)

scaler_poly = StandardScaler()
X_train_poly = scaler_poly.fit_transform(X_train_enc_poly)
X_valid_poly = scaler_poly.transform(X_valid_enc_poly)
X_test_poly  = scaler_poly.transform(X_test_enc_poly)

In [177]:
logreg_poly = LogisticRegression(random_state=21)
logreg_poly.fit(X_train_poly, y_train)

feature_names = X_train_enc_poly.columns.tolist()
coeffs_args_by_usefullness = np.argsort(abs(logreg_poly.coef_)[0])
feature_names_sorted = np.array(feature_names)[coeffs_args_by_usefullness]
best_features = feature_names_sorted[15:]

X_train_clean_df = X_train_enc_poly[best_features]
X_valid_clean_df = X_valid_enc_poly[best_features]
X_test_clean_df  = X_test_enc_poly[best_features]

scaler_clean = StandardScaler()
X_train_clean = scaler_clean.fit_transform(X_train_clean_df)
X_valid_clean = scaler_clean.transform(X_valid_clean_df)
X_test_clean  = scaler_clean.transform(X_test_clean_df)

In [178]:
mylogreg_std = SGDLogisticRegression(epoch=500)
mylogreg_std.fit(X_train_std, y_train)
proba_std = mylogreg_std.predict_proba(X_valid_std)[:, 1]
print('std:', gini(my_roc_auc(y_valid, proba_std)))

mylogreg_poly = SGDLogisticRegression(epoch=500)
mylogreg_poly.fit(X_train_poly, y_train)
proba_poly = mylogreg_poly.predict_proba(X_valid_poly)[:, 1]
print('poly:', gini(my_roc_auc(y_valid, proba_poly)))

mylogreg_clean = SGDLogisticRegression(epoch=500, l1=1)
mylogreg_clean.fit(X_train_clean, y_train)
proba_clean = mylogreg_clean.predict_proba(X_valid_clean)[:, 1]
print('clean', gini(my_roc_auc(y_valid, proba_clean)))

std: 0.3430065606069761
poly: 0.3456321193923868
clean 0.2968028890584602


In [179]:
param_grid = {'l1': [0.2, 0.5, 0.7, 0.9], 'epoch': [500]}
grdsrch = GridSearchCV(estimator=SGDLogisticRegression(epoch=500), param_grid=param_grid, cv=5, scoring='roc_auc')
grdsrch.fit(X_train_poly, y_train)
roc_auc = grdsrch.best_score_
print('best params:', grdsrch.best_params_)
print('best Gini:', gini(roc_auc))

best params: {'epoch': 500, 'l1': 0.2}
best Gini: 0.35481389826030063


### 10. Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitted or not? Explain.

In [180]:
mylogreg_std = SGDLogisticRegression(epoch=500)
mylogreg_std.fit(X_train_std, y_train)
proba_std = mylogreg_std.predict_proba(X_valid_std)[:, 1]
print('std', gini(my_roc_auc(y_valid, proba_std)))

mylogreg_poly = SGDLogisticRegression(epoch=500)
mylogreg_poly.fit(X_train_poly, y_train)
proba_poly = mylogreg_poly.predict_proba(X_valid_poly)[:, 1]
print('poly:', gini(my_roc_auc(y_valid, proba_poly)))

mylogreg_clean = SGDLogisticRegression(epoch=500, l1=1)
mylogreg_clean.fit(X_train_clean, y_train)
proba_clean = mylogreg_clean.predict_proba(X_valid_clean)[:, 1]
print('clean:', gini(my_roc_auc(y_valid, proba_clean)))

std 0.34251065646086354
poly: 0.3448539941005886
clean: 0.2965409435326578


In [181]:
mylogreg_std = SGDLogisticRegression(epoch=500)
mylogreg_std.fit(X_train_std, y_train)
proba_std = mylogreg_std.predict_proba(X_test_std)[:, 1]
print('STD   Gini:', gini(my_roc_auc(y_test, proba_std)))

mylogreg_poly = SGDLogisticRegression(epoch=500)
mylogreg_poly.fit(X_train_poly, y_train)
proba_poly = mylogreg_poly.predict_proba(X_test_poly)[:, 1]
print('POLY  Gini:', gini(my_roc_auc(y_test, proba_poly)))

mylogreg_clean = SGDLogisticRegression(epoch=500, l1=1)
mylogreg_clean.fit(X_train_clean, y_train)
proba_clean = mylogreg_clean.predict_proba(X_test_clean)[:, 1]
print('CLEAN Gini:', gini(my_roc_auc(y_test, proba_clean)))

STD   Gini: 0.17453763059548044
POLY  Gini: 0.17506750124940962
CLEAN Gini: 0.31255709813081256


##### here we can see a HUGE quality drop so i would tell that my model is overfitted.

### 11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics.
Compare your algorithms on the test dataset using AUC PR metric.

In [182]:
def recall(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()
    return TP / (TP + FN)

In [183]:
def precision(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return TP / (TP + FP)

In [184]:
def f1(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    return 2 * precision * recall / (precision + recall)

In [185]:
def auc_pr(y_true, y_pred_proba):
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    sorted_idx = np.argsort(y_pred_proba)[::-1]
    y_true_sorted = y_true[sorted_idx]

    TP = 0
    FP = 0
    FN = y_true.sum()
    points = []

    for i in y_true_sorted:
        if i == 1:
            TP+=1
            FN-=1
        else:
            FP+=1

        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        points.append((recall, precision))
    points = [(0.0, 1.0)] + points
    auc = 0
    for i in range(len(points) - 1):
        x1, y1 = points[i]
        x2,y2 = points[i+1]
        auc += (x2 - x1) * (y2 - y1) / 2
    return auc


In [186]:
y_pred_labels = mylogreg_poly.predict(X_test_poly)
y_pred_proba  = mylogreg_poly.predict_proba(X_test_poly)[:, 1]

my_precision = precision(y_test, y_pred_labels)
my_recall = recall(y_test, y_pred_labels)
my_f1 = f1(y_test, y_pred_labels)
my_auc_pr = auc_pr(y_test, y_pred_proba)

print(my_precision)
print(my_recall)
print(my_f1)
print(my_auc_pr)

0.0369935408103347
0.04245283018867924
0.039535613429557574
0.0001042527651737625


### Which hard label metric do you prefer for the task of detecting "lemon" cars?


##### recall